# here is a set of functions help the modification of the database a little bit easier.
Cinyu Zhu, Hopkins, June 2025

In [1]:
import numpy as np

In [2]:
def generate_sql_block(image_number: int, temperature: str) -> str:
    assert temperature in ["Low", "High"], "Temperature must be 'Low' or 'High'"
    
    table_name = "MODULE_UNDERGROUND"
    suffixes = ["Defects", "CTI_Visual", "Comments", "Reference", "Peak1", "Peak2", "Sigma", "Front"]
    amps = ["A", "B", "C", "D"]
    
    lines = [f"ALTER TABLE {table_name}"]
    
    for amp in amps:
        for suffix in suffixes:
            col_name = f"Image{image_number}_{temperature}_{suffix}_{amp}"
            lines.append(f"  ADD COLUMN `{col_name}` TEXT,")
    # Add the file field (only once at the end)
    lines.append(f"  ADD COLUMN `Image{image_number}_{temperature}_File` TEXT;")
    
    return "\n".join(lines)

# Example usage:


In [3]:
print(generate_sql_block(31, "Low"))


ALTER TABLE MODULE_UNDERGROUND
  ADD COLUMN `Image31_Low_Defects_A` TEXT,
  ADD COLUMN `Image31_Low_CTI_Visual_A` TEXT,
  ADD COLUMN `Image31_Low_Comments_A` TEXT,
  ADD COLUMN `Image31_Low_Reference_A` TEXT,
  ADD COLUMN `Image31_Low_Peak1_A` TEXT,
  ADD COLUMN `Image31_Low_Peak2_A` TEXT,
  ADD COLUMN `Image31_Low_Sigma_A` TEXT,
  ADD COLUMN `Image31_Low_Front_A` TEXT,
  ADD COLUMN `Image31_Low_Defects_B` TEXT,
  ADD COLUMN `Image31_Low_CTI_Visual_B` TEXT,
  ADD COLUMN `Image31_Low_Comments_B` TEXT,
  ADD COLUMN `Image31_Low_Reference_B` TEXT,
  ADD COLUMN `Image31_Low_Peak1_B` TEXT,
  ADD COLUMN `Image31_Low_Peak2_B` TEXT,
  ADD COLUMN `Image31_Low_Sigma_B` TEXT,
  ADD COLUMN `Image31_Low_Front_B` TEXT,
  ADD COLUMN `Image31_Low_Defects_C` TEXT,
  ADD COLUMN `Image31_Low_CTI_Visual_C` TEXT,
  ADD COLUMN `Image31_Low_Comments_C` TEXT,
  ADD COLUMN `Image31_Low_Reference_C` TEXT,
  ADD COLUMN `Image31_Low_Peak1_C` TEXT,
  ADD COLUMN `Image31_Low_Peak2_C` TEXT,
  ADD COLUMN `Image31_Low

In [4]:
import mysql.connector

def get_existing_columns(host, user, password, database, table):
    connection = mysql.connector.connect(
        host=host,
        user=user,
        password=password,
        database=database
    )
    cursor = connection.cursor()
    cursor.execute(f"SHOW COLUMNS FROM {table}")
    columns = {row[0] for row in cursor.fetchall()}
    cursor.close()
    connection.close()
    return columns

def generate_missing_columns(image_number: int, temperature: str, existing_columns: set):
    suffixes = ["Defects", "CTI_Visual", "Comments", "Reference", "Peak1", "Peak2", "Sigma", "Front"]
    sections = ["A", "B", "C", "D"]
    
    new_columns = []
    for section in sections:
        for suffix in suffixes:
            col_name = f"Image{image_number}_{temperature}_{suffix}_{section}"
            if col_name not in existing_columns:
                new_columns.append(col_name)

    # Add file field if needed
    file_col = f"Image{image_number}_{temperature}_File"
    if file_col not in existing_columns:
        new_columns.append(file_col)

    return new_columns

def add_columns_to_table(host, user, password, database, table, image_number, temperature):
    existing = get_existing_columns(host, user, password, database, table)
    new_columns = generate_missing_columns(image_number, temperature, existing)
    
    if not new_columns:
        print("-- All columns already exist. Nothing to add.")
        return
    
    sql = f"ALTER TABLE {table}\n" + ",\n".join(
        [f"  ADD COLUMN `{col}` TEXT" for col in new_columns]
    ) + ";"

    print("Executing SQL:\n", sql)

    connection = mysql.connector.connect(
        host=host,
        user=user,
        password=password,
        database=database
    )
    cursor = connection.cursor()
    cursor.execute(sql)
    connection.commit()
    cursor.close()
    connection.close()
    print(f"-- Successfully added {len(new_columns)} columns.")


In [5]:
## comment it out when unused
# # --- Replace with your credentials and inputs ---
# host = "localhost"
# user = "root"
# password = "MyLife4Aiur"
# database = "die_qc"
# table = "MODULE_UNDERGROUND"

# image_number = 99
# temperature = "Low"  # or "High"

# add_columns_to_table(host, user, password, database, table, image_number, temperature)


In [2]:
def generate_php_block(image_number: int, temperature: str, skip: str, binning: str,
                       resolution: str, exposure: str, aim: str) -> str:
    image_label = f"Image {image_number}"
    image_id = f"image{image_number}_{temperature.lower()}"
    description = f"{skip}skip, {binning}x binning, {resolution}, Active region, {exposure}s Exposure"

    return f"""<!-- {image_label} -  -->
<?php echo "<b>{image_label}, {temperature} Temp - [{description}] - Aim: {aim}</b>"; ?>
<form action="<?php echo $_SERVER['PHP_SELF']; ?>" method="post" enctype="multipart/form-data">
    <input type="hidden" name="id" value="<?php echo $id; ?>">
    <table border="1">
        <tr>
            <td align="left" style="width: 10%; white-space: nowrap;">Amplifier</td>
            <td align="left" style="width: 5%;">Defects?</td>
            <td align="left" style="width: 5%;">CTI? - Visual</td>
            <td align="left" style="width: 5%;">Energy Peak 1 [keV]</td>
            <td align="left" style="width: 5%;">Energy Peak 2 [keV]</td>
            <td align="left" style="width: 5%;">Sigma - Back Events [pixels]</td>
            <td align="left" style="width: 5%;">Front Events?</td>
            <td align="left" style="width: 25%;">Comments</td>
            <td align="left" style="width: 25%;">Reference Image</td>
        </tr>
        <?php
        $count = 0;
        $count_plus = 1;
        foreach ($ccds as $amp):
        ?>
        <tr>
            <td><?php echo "ch" . $count . " (ext" . $count_plus . ")"; ?></td>
            <td><?php generate_dropdown('{image_id}_defects_' . $amp, $yes_no_blank_array, ${{'{image_id}_defects_' . $amp}}); ?></td>
            <td><?php generate_dropdown('{image_id}_cti_visual_' . $amp, $yes_no_blank_array, ${{'{image_id}_cti_visual_' . $amp}}); ?></td>
            <td><input type="text" name="{image_id}_peak1_<?php echo $amp; ?>" value="<?php echo ${{'{image_id}_peak1_' . $amp}}; ?>" size="15"></td>
            <td><input type="text" name="{image_id}_peak2_<?php echo $amp; ?>" value="<?php echo ${{'{image_id}_peak2_' . $amp}}; ?>" size="15"></td>
            <td><input type="text" name="{image_id}_sigma_<?php echo $amp; ?>" value="<?php echo ${{'{image_id}_sigma_' . $amp}}; ?>" size="15"></td>
            <td><?php generate_dropdown('{image_id}_front_' . $amp, $yes_no_blank_array, ${{'{image_id}_front_' . $amp}}); ?></td>
            <td><input type="text" name="{image_id}_comments_<?php echo $amp; ?>" value="<?php echo ${{'{image_id}_comments_' . $amp}}; ?>" size="40"></td>
            <td><input type="text" name="{image_id}_reference_<?php echo $amp; ?>" value="<?php echo ${{'{image_id}_reference_' . $amp}}; ?>" size="50"></td>
        </tr>
        <?php $count++; $count_plus++; endforeach; ?>

        <tr>
            <td align="left" colspan="9" style="border: none; white-space: nowrap;">
                <input type="submit" value="Submit">
                <?php
		        $module_underground_id = isset($_SESSION['choosen_module_underground']) ? $_SESSION['choosen_module_underground'] : 0;
                $upload_dir = "/home/uploads/edit_module_underground/module_underground_" . $module_underground_id . "/";
                $base_url = "/uploads/edit_module_underground/module_underground_" . $module_underground_id . "/";

                ${{'{image_id}_file_name'}} = "{image_id}_file.png";
                ${{'{image_id}_file_path'}} = $upload_dir . ${{'{image_id}_file_name'}};
                ${{'{image_id}_log_name'}} = "{image_id}_log.log";
                ${{'{image_id}_log_path'}} = $upload_dir . ${{'{image_id}_log_name'}};

                if (file_exists(${{'{image_id}_file_path'}}) && !isset($_SESSION['file_url_' . $module_underground_id]['{image_id}_file'])) {{
                    $_SESSION['file_url_' . $module_underground_id]['{image_id}_file'] = $base_url . ${{'{image_id}_file_name'}};
                }}
                if (!empty(${{'{image_id}_file_name'}}) && file_exists(${{'{image_id}_file_path'}})) {{
                    $file_exists = true;
                    $file_url = $_SESSION['file_url_' . $module_underground_id]['{image_id}_file'];
                }} else {{
                    $file_exists = false;
                }}

                if (file_exists(${{'{image_id}_log_path'}}) && !isset($_SESSION['log_url_' . $module_underground_id]['{image_id}_log'])) {{
                    $_SESSION['log_url_' . $module_underground_id]['{image_id}_log'] = $base_url . ${{'{image_id}_log_name'}};
                }}
                if (!empty(${{'{image_id}_log_name'}}) && file_exists(${{'{image_id}_log_path'}})) {{
                    $log_exists = true;
                    $log_url = $_SESSION['log_url_' . $module_underground_id]['{image_id}_log'];
                }} else {{
                    $log_exists = false;
                }}
                ?>
                <?php if ($file_exists): ?>
                    <a href="<?php echo htmlspecialchars($file_url); ?>" target="_blank">
                        <img src="pixmaps/icon.png" alt="{image_label}_{temperature} File" style="height: 20px;">
                    </a>
                <?php endif; ?>
                <label for="{image_id}_file">Image File:</label>
                <input type="file" name="{image_id}_file" accept="image/png, image/jpeg, application/pdf">

                <?php if ($log_exists): ?>
                    <a href="<?php echo htmlspecialchars($log_url); ?>" target="_blank">
                        <img src="pixmaps/icon2.png" alt="{image_label}_{temperature} Log" style="height: 20px;">
                    </a>
                <?php endif; ?>
                <label for="{image_id}_log">Log File:</label>
                <input type="file" name="{image_id}_log" accept=".log,text/plain">
            </td>
        </tr>
    </table>
</form>
<br><br>
<!-- Insert the code block of next image after this -->
"""


In [6]:
print(generate_php_block(
    image_number=7,
    temperature="Low",
    skip="500",
    binning="1x10",
    resolution="640rx320c",
    exposure="500",
    aim="High Resolution Fe55 Cluster Analysis, CTI, Noise"
))

<!-- Image 7 -  -->
<?php echo "<b>Image 7, Low Temp - [500skip, 1x10x binning, 640rx320c, Active region, 500s Exposure] - Aim: High Resolution Fe55 Cluster Analysis, CTI, Noise</b>"; ?>
<form action="<?php echo $_SERVER['PHP_SELF']; ?>" method="post" enctype="multipart/form-data">
    <input type="hidden" name="id" value="<?php echo $id; ?>">
    <table border="1">
        <tr>
            <td align="left" style="width: 10%; white-space: nowrap;">Amplifier</td>
            <td align="left" style="width: 5%;">Defects?</td>
            <td align="left" style="width: 5%;">CTI? - Visual</td>
            <td align="left" style="width: 5%;">Energy Peak 1 [keV]</td>
            <td align="left" style="width: 5%;">Energy Peak 2 [keV]</td>
            <td align="left" style="width: 5%;">Sigma - Back Events [pixels]</td>
            <td align="left" style="width: 5%;">Front Events?</td>
            <td align="left" style="width: 25%;">Comments</td>
            <td align="left" style="width: 25%